# Mechanism support table

Regenerates the compact diagnostics table used in `final_report.tex` from cached CSV artifacts. This notebook is CPU-only and does not rerun model training.

In [1]:

from pathlib import Path
import pandas as pd

root = Path('..').resolve()
coat_diag = pd.read_csv(root / 'artifacts' / 'summaries' / 'mechanism_diagnostics.csv')
swap = pd.read_csv(root / 'artifacts' / 'summaries' / 'swap_summary.csv')
yahoo = pd.read_csv(root / 'artifacts' / 'yahoo_full' / 'summaries' / 'all_runs.csv')

get = lambda name: float(coat_diag.loc[coat_diag['diagnostic'] == name, 'value'].iloc[0])
yahoo_k1 = yahoo.loc[yahoo['model'] == 'lightgcn_k1'].iloc[0]
yahoo_k3 = yahoo.loc[yahoo['model'] == 'lightgcn_k3'].iloc[0]
swap_a = swap.loc[swap['comparison'] == 'corrected_k2'].iloc[0]

rows = [
    (
        'Yahoo depth shift',
        'Avg. train degree in top-5, K1 $\\to$ K3',
        f"{yahoo_k1['avg_recommended_train_degree']:.1f} $\\to$ {yahoo_k3['avg_recommended_train_degree']:.1f}",
    ),
    (
        'Coat popularity signal',
        'Spearman(train degree, random positive rate)',
        f"{get('degree_vs_random_positive_rate_spearman'):.3f}",
    ),
    (
        'Coat head/tail preference',
        'Random positive rate, tail $\\to$ head quartile',
        f"{get('tail_quartile_random_positive_rate'):.3f} $\\to$ {get('head_quartile_random_positive_rate'):.3f}",
    ),
    (
        'Correction behavior',
        'Avg. degree, vanilla-only $\\to$ correction-only swaps',
        f"{swap_a['vanilla_only_degree']:.2f} $\\to$ {swap_a['correction_only_degree']:.2f}",
    ),
    (
        'Correction error mode',
        'Positive rate, vanilla-only $\\to$ correction-only swaps',
        f"{swap_a['vanilla_only_label_rate']:.3f} $\\to$ {swap_a['correction_only_label_rate']:.3f}",
    ),
]

table = pd.DataFrame(rows, columns=['Claim', 'Diagnostic', 'Value'])
print(table.to_string(index=False))

latex_lines = [
    r'\begin{table}[!h]',
    r'\centering',
    r'\scriptsize',
    r'\caption{Mechanism diagnostics from cached artifacts. Yahoo degree values use the full-data run; Coat diagnostics use the 20-seed artifacts.}',
    r'\label{tab:mechanism}',
    r'\begin{tabular}{@{}p{0.20\linewidth}p{0.56\linewidth}r@{}}',
    r'\toprule',
    r'Claim & Diagnostic & Value \\',
    r'\midrule',
]
for claim, diagnostic, value in rows:
    latex_lines.append(f'{claim} & {diagnostic} & {value} \\\\')
latex_lines += [
    r'\bottomrule',
    r'\end{tabular}',
    r'\end{table}',
]
latex = '\n'.join(latex_lines) + '\n'
out = root / 'tables' / 'mechanism_support.tex'
out.write_text(latex)
print(f'\nWrote {out}')
print(latex)


                    Claim                                              Diagnostic             Value
        Yahoo depth shift                 Avg. train degree in top-5, K1 $\to$ K3 265.7 $\to$ 284.4
   Coat popularity signal            Spearman(train degree, random positive rate)             0.191
Coat head/tail preference          Random positive rate, tail $\to$ head quartile 0.364 $\to$ 0.443
      Correction behavior   Avg. degree, vanilla-only $\to$ correction-only swaps 12.53 $\to$ 10.04
    Correction error mode Positive rate, vanilla-only $\to$ correction-only swaps 0.447 $\to$ 0.401

Wrote /Users/lam91/Downloads/CPSC_483_finalproject_20260425/reproduceable_workspace/tables/mechanism_support.tex
\begin{table}[!h]
\centering
\scriptsize
\caption{Mechanism diagnostics from cached artifacts. Yahoo degree values use the full-data run; Coat diagnostics use the 20-seed artifacts.}
\label{tab:mechanism}
\begin{tabular}{@{}p{0.20\linewidth}p{0.56\linewidth}r@{}}
\toprule
Claim & Diagn